# Decision Tree Study

This notebook studies decision trees from the simplest one-feature split to a multi-feature model for the Home Credit problem.

Learning sequence:
1. Calculate impurity for a small example.
2. Compare candidate thresholds.
3. Choose the best split.
4. Create leaf probabilities.
5. Study stopping rules and overfitting.
6. Compare a scratch tree with scikit-learn.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Decision-tree study notebook is ready.")


Decision-tree study notebook is ready.


## First experiment

Begin with one feature and a tiny dataset. Calculate Gini impurity manually before using the full Home Credit data.


## Load the training tables

We load the base application table and both `train_static_0` partitions, then merge them using the unique `case_id`.


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "parquet_files" / "train").is_dir()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Open Jupyter from the project folder or its notebooks folder.")

train_dir = project_dir / "data" / "parquet_files" / "train"
train_base_path = train_dir / "train_base.parquet"
static_0_0_path = train_dir / "train_static_0_0.parquet"
static_0_1_path = train_dir / "train_static_0_1.parquet"

for required_path in [train_base_path, static_0_0_path, static_0_1_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")

train_base_df = pd.read_parquet(train_base_path)
train_static_0_0_df = pd.read_parquet(static_0_0_path)
train_static_0_1_df = pd.read_parquet(static_0_1_path)

if list(train_static_0_0_df.columns) != list(train_static_0_1_df.columns):
    raise ValueError("The two static_0 partitions do not have identical columns.")

train_static_0_df = pd.concat(
    [train_static_0_0_df, train_static_0_1_df],
    ignore_index=True,
)

print("train_base shape:", train_base_df.shape)
print("train_static_0_0 shape:", train_static_0_0_df.shape)
print("train_static_0_1 shape:", train_static_0_1_df.shape)
print("combined train_static_0 shape:", train_static_0_df.shape)
display(train_base_df.head())
display(train_static_0_df.head())


train_base shape: (1526659, 5)
train_static_0_0 shape: (1003757, 168)
train_static_0_1 shape: (522902, 168)
combined train_static_0 shape: (1526659, 168)


,case_id,date_decision,MONTH,WEEK_NUM,target
0,0,2019-01-03,201901,0,0
1,1,2019-01-03,201901,0,0
2,2,2019-01-04,201901,0,0
3,3,2019-01-03,201901,0,0
4,4,2019-01-04,201901,0,1


,case_id,actualdpdtolerance_344P,amtinstpaidbefduel24m_4187115A,annuity_780A,annuitynextmonth_57A,applicationcnt_361L,applications30d_658L,applicationscnt_1086L,applicationscnt_464L,applicationscnt_629L,...,sellerplacecnt_915L,sellerplacescnt_216L,sumoutstandtotal_3546847A,sumoutstandtotalest_4493215A,totaldebt_9A,totalsettled_863A,totinstallast1m_4525188A,twobodfilling_608L,typesuite_864L,validfrom_1069D
0,0,NaN,NaN,1917.6,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,BO,None,None
1,1,NaN,NaN,3134.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,BO,None,None
2,2,NaN,NaN,4937.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,BO,AL,None
3,3,NaN,NaN,4643.6,0.0,0.0,1.0,0.0,2.0,0.0,...,1.0,1.0,NaN,NaN,0.0,0.0,NaN,BO,AL,None
4,4,NaN,NaN,3390.2,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,BO,AL,None


## Merge base and static features

The base table supplies the target and time information. The static table supplies the application features.


In [ ]:
base_labels_df = train_base_df[["case_id", "WEEK_NUM", "target"]].copy()

if train_base_df["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values exist in train_base_df.")
if train_static_0_df["case_id"].duplicated().any():
    raise ValueError("Duplicate case_id values exist in combined train_static_0_df.")

decision_tree_modeling_df = base_labels_df.merge(
    train_static_0_df,
    on="case_id",
    how="inner",
    validate="one_to_one",
)

if len(decision_tree_modeling_df) != len(train_base_df):
    raise ValueError("The merge lost applications compared with train_base_df.")
if decision_tree_modeling_df["target"].isna().any():
    raise ValueError("Merged target contains missing values.")

print("Merged decision-tree table shape:", decision_tree_modeling_df.shape)
print("Merged unique case IDs:", decision_tree_modeling_df["case_id"].nunique())
display(decision_tree_modeling_df.head())


## Leakage-safe preprocessing for a decision tree

Decision trees do not require feature scaling because they compare values with thresholds. We still need to impute missing values and encode categorical columns. All preprocessing is fitted on weeks 0–51 only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

decision_tree_train_df = decision_tree_modeling_df[decision_tree_modeling_df["WEEK_NUM"] <= 51].copy()
decision_tree_validation_df = decision_tree_modeling_df[decision_tree_modeling_df["WEEK_NUM"].between(52, 71)].copy()
decision_tree_holdout_df = decision_tree_modeling_df[decision_tree_modeling_df["WEEK_NUM"].between(72, 91)].copy()

decision_tree_excluded_columns = {"case_id", "WEEK_NUM", "target"}
decision_tree_feature_columns = [
    column for column in decision_tree_modeling_df.columns
    if column not in decision_tree_excluded_columns
]
decision_tree_numeric_columns = [
    column for column in decision_tree_feature_columns
    if pd.api.types.is_numeric_dtype(decision_tree_train_df[column])
    and not pd.api.types.is_bool_dtype(decision_tree_train_df[column])
]
decision_tree_categorical_columns = [
    column for column in decision_tree_feature_columns
    if column not in decision_tree_numeric_columns
]

decision_tree_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
decision_tree_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("to_string", FunctionTransformer(lambda values: values.astype(str), feature_names_out="one-to-one")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype="float32")),
])

decision_tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", decision_tree_numeric_pipeline, decision_tree_numeric_columns),
        ("categorical", decision_tree_categorical_pipeline, decision_tree_categorical_columns),
    ],
    remainder="drop",
)

X_tree_train = decision_tree_preprocessor.fit_transform(decision_tree_train_df[decision_tree_feature_columns])
X_tree_validation = decision_tree_preprocessor.transform(decision_tree_validation_df[decision_tree_feature_columns])
X_tree_holdout = decision_tree_preprocessor.transform(decision_tree_holdout_df[decision_tree_feature_columns])
y_tree_train = decision_tree_train_df["target"].to_numpy(dtype=np.int64)
y_tree_validation = decision_tree_validation_df["target"].to_numpy(dtype=np.int64)
y_tree_holdout = decision_tree_holdout_df["target"].to_numpy(dtype=np.int64)
tree_feature_names = list(decision_tree_preprocessor.get_feature_names_out())

for matrix_name, matrix in {
    "X_tree_train": X_tree_train,
    "X_tree_validation": X_tree_validation,
    "X_tree_holdout": X_tree_holdout,
}.items():
    matrix_values = matrix.data if hasattr(matrix, "data") and not isinstance(matrix, np.ndarray) else matrix
    if not np.isfinite(matrix_values).all():
        raise ValueError(f"{matrix_name} contains non-finite values.")

print("Decision-tree training rows:", len(decision_tree_train_df))
print("Decision-tree validation rows:", len(decision_tree_validation_df))
print("Decision-tree holdout rows:", len(decision_tree_holdout_df))
print("Numeric input columns:", len(decision_tree_numeric_columns))
print("Categorical input columns:", len(decision_tree_categorical_columns))
print("Prepared feature columns:", len(tree_feature_names))
print("Training matrix shape:", X_tree_train.shape)
print("Validation matrix shape:", X_tree_validation.shape)
print("Holdout matrix shape:", X_tree_holdout.shape)
print("No scaling was applied.")
